# CAID Benchmark — Drive-safe Run

Runtime → Run all

Data is written directly to Google Drive. If runtime disconnects, data survives.

## 1. Mount Drive (mandatory)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/caid_benchmark_data'
os.makedirs(DRIVE_DATA, exist_ok=True)
print(f'Drive ready: {DRIVE_DATA}')
print(f'Existing runs: {os.listdir(DRIVE_DATA)}')

## 2. Upload archive and install

In [ ]:
from google.colab import files
import os, shutil
if os.path.exists('/content/caid'):
    shutil.rmtree('/content/caid')
os.makedirs('/content/caid', exist_ok=True)
print('Choose caid_code.zip')
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.endswith('.zip')), None)
if not zip_name:
    raise RuntimeError('No .zip uploaded')
shutil.move(zip_name, f'/content/{zip_name}')
%cd /content/caid
!unzip -q /content/{zip_name}
!pip install -q -r requirements.txt
!chmod +x run_full_pipeline.sh

# Symlink data/ to Drive — all writes go straight there
import os
if os.path.lexists('/content/caid/data'):
    if os.path.islink('/content/caid/data'):
        os.unlink('/content/caid/data')
    else:
        import shutil
        shutil.rmtree('/content/caid/data')
os.symlink('/content/drive/MyDrive/caid_benchmark_data', '/content/caid/data')
print(f'data/ -> Drive: {os.readlink("/content/caid/data")}')
print()
print('Verifying classifier...')
!python src/test_classifier.py | tail -3

## 3. API keys

In [ ]:
import os
from getpass import getpass
providers = [
    ('GROQ_API_KEY', 'Groq', 'gsk_...'),
    ('OPENROUTER_API_KEY', 'OpenRouter', 'sk-or-v1-...'),
    ('CEREBRAS_API_KEY', 'Cerebras', 'csk-...'),
    ('GOOGLE_API_KEY', 'Google AI Studio', 'AIza...'),
    ('HF_TOKEN', 'HuggingFace', 'hf_...'),
]
for env_var, name, hint in providers:
    val = getpass(f'{name} ({hint}): ').strip()
    if val:
        os.environ[env_var] = val
        print(f'  {name}: set')
    else:
        print(f'  {name}: skipped')
active = [k for k in ['GROQ_API_KEY', 'OPENROUTER_API_KEY', 'CEREBRAS_API_KEY', 'GOOGLE_API_KEY', 'HF_TOKEN'] if os.environ.get(k)]
print(f'\nActive: {active}')
if not active:
    raise RuntimeError('No keys provided.')

## 4. Run benchmark (data goes straight to Drive)

In [ ]:
%cd /content/caid
import os
RUN_ID = 'run_' + os.popen('date +%Y%m%d_%H%M').read().strip()
print(f'Run ID: {RUN_ID}')
print(f'Will write to: /content/drive/MyDrive/caid_benchmark_data/raw/{RUN_ID}/')
print()
!python src/run_benchmark.py --all --conditions vendor,none --n 3 \
  --pace-min 4 --pace-max 6 --shuffle-models \
  --run-id {RUN_ID}
with open('/content/drive/MyDrive/caid_benchmark_data/last_run_id.txt', 'w') as f:
    f.write(RUN_ID)

## 5. Analysis

In [ ]:
%cd /content/caid
RUN_ID = open('/content/drive/MyDrive/caid_benchmark_data/last_run_id.txt').read().strip()
print(f'Analysing: {RUN_ID}')
print()
!python src/analyze.py --run-id {RUN_ID}
import pandas as pd
df = pd.read_csv(f'data/raw/{RUN_ID}/metrics_per_model.csv')
df.sort_values('overall_rate', ascending=False)